# Evaluate ranges of output dataset
Diagnostics comparing our downscaled data against the ranges from ERA5. Rather than hard-and-fast checks for aphysical values (e.g. negative precip), this checks for squishier situations like is there a high precipitation event in the Sahara. Goal is to surface problems. 

## running this notebook

```Python
uv run coiled notebook start --vm-type m8g.16xlarge --region 'us-west-2' --tag Project=SRM --sync
```

even with these resources, this will still take ~an hour to run so, grab some coffee.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import xarray as xr
from dask.distributed import Client

from srm import catalog
from srm.utils import open_icechunk, resolve_s3_glob

In [ ]:
client = Client()
client

In [ ]:
def get_fname(root_path, var, scenario, version="test015_benchmark", ens="003", scenario2=None):
    if scenario2 is None:
        scenario2 = scenario
    fname = (
        root_path
        + version
        + "/"
        + scenario
        + "/CESM2-WACCM/"
        + var
        + "/"
        + ens
        + "/global/era5/*/"
        + scenario2
        + ".icechunk/"
    )
    return fname

In [ ]:
_MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Grey for zero, continuous Reds for positive values
_EXCEEDANCE_CMAP = plt.cm.Reds.copy()
_EXCEEDANCE_CMAP.set_under("lightgrey")


def calc_era5_threshold(era5, var, direction, period="annual", downscaled_da=None):
    """Per-pixel ERA5 threshold: 'gt' → max, 'lt' → min; 'monthly' groups by calendar month."""
    if downscaled_da is not None:
        era5_da = era5[var].sel(lat=downscaled_da.lat, lon=downscaled_da.lon, method="nearest")
    else:
        era5_da = era5[var]
    if period == "annual":
        if direction == "gt":
            return era5_da.max(dim="time").load()
        elif direction == "lt":
            return era5_da.min(dim="time").load()
    elif period == "monthly":
        if direction == "gt":
            return era5_da.groupby("time.month").max(dim="time").load()
        elif direction == "lt":
            return era5_da.groupby("time.month").min(dim="time").load()


def count_exceedances(downscaled_da, threshold, direction, period="annual"):
    """Annual: (lat, lon) count over all time. Monthly: (month, lat, lon) count per calendar month."""
    if period == "annual":
        if direction == "gt":
            return (downscaled_da > threshold).sum(dim="time").load()
        return (downscaled_da < threshold).sum(dim="time").load()
    counts = []
    for m, group in downscaled_da.groupby("time.month"):
        thresh = threshold.sel(month=m)
        exceeded = group > thresh if direction == "gt" else group < thresh
        counts.append(exceeded.sum("time").expand_dims(month=[m]))
    return xr.concat(counts, dim="month").load()


def plot_exceedance_map(count_da, title, period="annual", var_lims=None):
    """Single panel for annual; 3×4 panel for monthly (one per calendar month)."""
    # vmin=0.5 pushes exactly-zero cells below the colormap range → rendered as lightgrey
    vmin = 0.5 if var_lims is None else max(var_lims[0], 0.5)
    vmax = None if var_lims is None else var_lims[1]

    def _plot_panel(da, ax):
        da.plot(
            ax=ax,
            cmap=_EXCEEDANCE_CMAP,
            vmin=vmin,
            vmax=vmax,
            transform=ccrs.PlateCarree(),
        )
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)

    if period == "annual":
        fig, ax = plt.subplots(figsize=(8, 6), subplot_kw=dict(projection=ccrs.PlateCarree()))
        _plot_panel(count_da, ax)
        ax.set_title(title)
        plt.tight_layout()
        plt.show()
    elif period == "monthly":
        fig, axs = plt.subplots(
            nrows=3, ncols=4, figsize=(24, 12), subplot_kw=dict(projection=ccrs.PlateCarree())
        )
        fig.suptitle(title, fontsize=12)
        for i, ax in enumerate(axs.flatten()):
            m = i + 1
            _plot_panel(count_da.sel(month=m), ax)
            ax.set_title(_MONTH_NAMES[i])
        plt.tight_layout()
        plt.show()

In [ ]:
# Each entry is var: [(direction, period, title)]
checks_dict = {
    "tasmax": [
        ("gt", "annual", "Greater than ERA5 annual max tasmax"),
        ("gt", "monthly", "Greater than ERA5 monthly max tasmax"),
    ],
    "tasmin": [
        ("lt", "annual", "Less than ERA5 annual min tasmin"),
        ("lt", "monthly", "Less than ERA5 monthly min tasmin"),
    ],
    "tas": [
        ("gt", "annual", "Greater than ERA5 annual max tasmean"),
        ("lt", "annual", "Less than ERA5 annual min tasmean"),
    ],
    "pr": [
        ("gt", "monthly", "Greater than ERA5 monthly max precipitation"),
        ("gt", "annual", "Greater than ERA5 annual max precipitation"),
    ],
    "rsds": [("gt", "annual", "Greater than ERA5 annual max rsds"),
            ("lt", "annual", "Less than ERA5 annual min rsds"),],
    "hurs": [("lt", "annual", "Less than ERA5 annual min relative humidity")],
}

In [ ]:
recalculate_era5_stats = False
if recalculate_era5_stats:
    era5 = catalog.get("ERA5").to_xarray()
    era5_stats_ds = xr.Dataset()
    for var, checks in checks_dict.items():
        for direction, period, title in checks:
            era5_stats_ds["-".join([var, direction, period])] = calc_era5_threshold(
                era5, var, direction, period=period
            )
    
            era5_stats_ds.to_zarr("s3://carbonplan-scratch/srm/output/qa/era5_stats.zarr", mode="w")
else:
    era5_stats_ds = xr.open_zarr("s3://carbonplan-scratch/srm/output/qa/era5_stats.zarr").load()

In [ ]:
root_path = "s3://carbonplan-srm/output/production/"
version = "v2026.06.22"

In [ ]:
setups = [
    ("historical", "r3i1p1f1", ["tas", "pr", "rsds", "hurs"]),
    ("g6-1.5k", "003", ["tas", "pr", "rsds", "hurs"]),
    ("ssp245", "003", ["tas", "pr", "rsds", "hurs"]),
]
for scenario, ens, variables in setups:
    for var in variables:
        checks = checks_dict[var]
        for direction, period, title in checks:
            fname = get_fname(
                root_path=root_path, var=var, scenario=scenario, version=version, ens=ens
            )
            fname = resolve_s3_glob(fname)
            downscaled_da = open_icechunk(path=fname)[var]
            threshold = era5_stats_ds["-".join([var, direction, period])]
            count = count_exceedances(downscaled_da, threshold, direction, period)
            plot_exceedance_map(
                count, title + f"\nscenario: {scenario}, v: {version}, ens: {ens}", period=period
            )

In [ ]:
setups = [
    ("historical", "r3i1p1f1", ["tas", "pr", "rsds", "hurs"]),
    ("g6-1.5k", "003", ["tas", "pr", "rsds", "hurs"]),
    ("ssp245", "003", ["tas", "pr", "rsds", "hurs"]),
]
for scenario, ens, variables in setups:
    for var in variables:
        fname = get_fname(root_path=root_path, var=var, scenario=scenario, version=version, ens=ens)
        fname = resolve_s3_glob(fname)
        downscaled_da = open_icechunk(path=fname)[var]
        fig, ax = plt.subplots(figsize=(8, 6), subplot_kw=dict(projection=ccrs.PlateCarree()))
        downscaled_da.min(dim="time").plot(ax=ax)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        plt.title(var + " " + scenario + " " + ens + " time minimum")
        plt.show()

In [ ]:
setups = [
    ("historical", "r3i1p1f1", ["tas", "pr", "rsds", "hurs"]),
    ("g6-1.5k", "003", ["tas", "pr", "rsds", "hurs"]),
    ("ssp245", "003", ["tas", "pr", "rsds", "hurs"]),
]
for scenario, ens, variables in setups:
    for var in variables:
        fname = get_fname(root_path=root_path, var=var, scenario=scenario, version=version, ens=ens)
        fname = resolve_s3_glob(fname)
        downscaled_da = open_icechunk(path=fname)[var]
        fig, ax = plt.subplots(figsize=(8, 6), subplot_kw=dict(projection=ccrs.PlateCarree()))
        downscaled_da.max(dim="time").plot(ax=ax)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        plt.title(var + " " + scenario + " " + ens + " time maximum")
        plt.show()

Repeat the time max/min plots but using robust=True in the plotting to look for any artifacts

In [ ]:
setups = [
    ("historical", "r3i1p1f1", ["tas", "pr", "rsds", "hurs"]),
    ("g6-1.5k", "003", ["tas", "pr", "rsds", "hurs"]),
    ("ssp245", "003", ["tas", "pr", "rsds", "hurs"]),
]

for scenario, ens, variables in setups:
    for var in variables:
        fname = get_fname(root_path=root_path, var=var, scenario=scenario, version=version, ens=ens)
        fname = resolve_s3_glob(fname)
        downscaled_da = open_icechunk(path=fname)[var]
        fig, ax = plt.subplots(figsize=(8, 6), subplot_kw=dict(projection=ccrs.PlateCarree()))
        downscaled_da.min(dim="time").plot(ax=ax, robust=True)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        plt.title(var + " " + scenario + " " + ens + " time minimum")
        plt.show()

In [ ]:
setups = [
    ("historical", "r3i1p1f1", ["tas", "pr", "rsds", "hurs"]),
    ("g6-1.5k", "003", ["tas", "pr", "rsds", "hurs"]),
    ("ssp245", "003", ["tas", "pr", "rsds", "hurs"]),
]

for scenario, ens, variables in setups:
    for var in variables:
        fname = get_fname(root_path=root_path, var=var, scenario=scenario, version=version, ens=ens)
        fname = resolve_s3_glob(fname)
        downscaled_da = open_icechunk(path=fname)[var]
        fig, ax = plt.subplots(figsize=(8, 6), subplot_kw=dict(projection=ccrs.PlateCarree()))
        downscaled_da.max(dim="time").plot(ax=ax, robust=True)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        plt.title(var + " " + scenario + " " + ens + " time maximum")
        plt.show()